<a href="https://colab.research.google.com/github/leee-beep/enterprise-rag-prototype/blob/main/notebooks/01_rag_basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 585.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 5.3 MB/s eta 0:00:00


In [2]:
import os
from getpass import getpass

api_key = getpass("請貼上 Gemini API Key：")
os.environ["GEMINI_API_KEY"] = api_key

print("API Key 已暫時載入。")

請貼上 Gemini API Key：··········
API Key 已暫時載入。


In [3]:
from google import genai

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

print("適合文字生成的 Flash 模型：")

for model in client.models.list():
    name = getattr(model, "name", "")
    supported_actions = getattr(model, "supported_actions", []) or []

    if (
        "generateContent" in supported_actions
        and "gemini" in name.lower()
        and "flash" in name.lower()
        and "embedding" not in name.lower()
        and "image" not in name.lower()
        and "live" not in name.lower()
    ):
        print("-", name)

適合文字生成的 Flash 模型：
- models/gemini-2.5-flash
- models/gemini-2.0-flash
- models/gemini-2.0-flash-001
- models/gemini-2.0-flash-lite-001
- models/gemini-2.0-flash-lite
- models/gemini-2.5-flash-preview-tts
- models/gemini-flash-latest
- models/gemini-flash-lite-latest
- models/gemini-2.5-flash-lite
- models/gemini-3-flash-preview
- models/gemini-3.1-flash-lite-preview
- models/gemini-3.1-flash-lite
- models/gemini-3.5-flash
- models/gemini-omni-flash-preview
- models/gemini-3.1-flash-tts-preview


In [4]:
model_name = "gemini-3.1-flash-lite"

response = client.models.generate_content(
    model=model_name,
    contents="請用繁體中文，用兩句話解釋什麼是 RAG。"
)

print(response.text)

RAG（檢索增強生成）是一種透過檢索外部資料庫資訊，來輔助 AI 模型生成更精準內容的技術。這種方法能有效降低 AI 的幻覺問題，並確保回答內容基於最新且可靠的特定知識來源。


In [5]:
!pip install -q langchain langchain-text-splitters

In [6]:
text = """
AI 部門新人培訓手冊（Version 1.0）
第一章 公司 AI 部門介紹
歡迎加入 AI 研發部門。
本部門主要負責研究並開發企業內部 AI 應用，協助公司提升工作效率、降低重複性工作以及建立智慧化決策系統。目前部門的研究方向包括 Retrieval-Augmented Generation（RAG）、AI Agent、企業知識管理、資料分析、自動化流程以及大型語言模型（Large Language Models, LLM）。
本部門希望建立一套能夠協助員工快速查詢公司知識、分析文件並回答問題的 AI 系統，因此新人需要了解 RAG 的基本概念以及相關技術。

第二章 RAG 是什麼？
RAG（Retrieval-Augmented Generation）中文通常翻譯為「檢索增強生成」。
傳統的大型語言模型只能根據訓練資料回答問題，因此容易出現資訊過時或產生幻覺（Hallucination）的情況。
RAG 的核心概念是：
當使用者提出問題時，系統會先到知識庫搜尋最相關的文件，再將搜尋結果提供給大型語言模型，由模型根據這些資料回答問題，而不是直接依靠模型本身的記憶。
因此，RAG 可以回答公司內部文件、產品規格、技術文件以及 SOP 等內容。

第三章 RAG 的工作流程
一套完整的 RAG 系統通常包含以下步驟：
第一步，使用者將文件上傳至知識庫，例如 PDF、Word 或 PowerPoint。
第二步，系統會解析文件內容，並將文件切割成多個 Chunk。
第三步，每一個 Chunk 都會透過 Embedding Model 轉換成向量（Vector）。
第四步，向量會儲存在 Vector Database 中。
第五步，當使用者提出問題時，Retriever 會搜尋最相關的 Chunk。
最後，大型語言模型（LLM）會根據 Retriever 找到的內容產生回答。

第四章 AI Agent
AI Agent 與一般聊天機器人最大的不同，在於 Agent 可以主動完成任務。
例如：
查詢公司資料
搜尋網路資訊
撰寫報告
呼叫 API
使用工具
執行程式
Agent 可以根據使用者的需求，自行決定需要哪些工具，再逐步完成任務，因此比單純聊天模型具有更高的自主性。

第五章 Chunk 與 Embedding
Chunk 是指將長篇文件切割成較小的內容片段。
由於大型語言模型一次能處理的文字長度有限，因此 RAG 不會直接把一本 300 頁的文件交給模型，而是先切割成許多 Chunk。
Embedding 則是將每一個 Chunk 轉換成數值向量。
這些向量可以表示文字的語意，因此即使使用不同的文字詢問相同意思的問題，系統仍然可以找到最相關的內容。

第六章 Retriever
Retriever 的主要工作是搜尋知識庫。
例如：
使用者詢問：
公司的請假流程是什麼？
Retriever 不會直接回答，而是先到 Vector Database 搜尋最相關的 Chunk。
找到最相關的內容後，再交給大型語言模型整理答案。
Retriever 的品質會直接影響整套 RAG 系統的回答品質。

第七章 公司資訊安全規範
所有員工都應遵守公司資訊安全政策。
未經主管同意，不得將公司機密文件上傳至外部平台。
不得將 API Key、密碼、Token 或公司帳號分享給其他人。
使用 AI 工具時，應避免輸入客戶個人資料、財務資料或未公開資訊。
若需要使用外部 AI 平台進行測試，應先確認資料已完成匿名化，並符合公司資訊安全規範。

第八章 AI 專案開發流程
一個企業 AI 專案通常包含以下流程：
確認需求。
蒐集資料。
建立知識庫。
文件解析。
建立 Embedding。
建立 Vector Database。
建立 Retriever。
選擇大型語言模型。
建立 Prompt。
建立 AI Assistant。
測試回答品質。
持續優化 Chunk、Embedding、Retriever 與 Prompt。

第九章 常見問題（FAQ）
Q1：RAG 與 ChatGPT 有什麼不同？
A：ChatGPT 主要依靠模型本身的知識回答問題；RAG 則會先搜尋知識庫，再根據搜尋結果回答，因此更適合企業內部文件查詢。

Q2：Chunk 越大越好嗎？
A：不一定。Chunk 過大可能包含過多無關資訊；Chunk 過小則可能缺乏完整上下文，因此需要依照文件類型調整。

Q3：Embedding 的用途是什麼？
A：Embedding 將文字轉換為向量，方便系統比較語意相似度，提升搜尋品質。

Q4：Retriever 與 LLM 有什麼差別？
A：Retriever 負責搜尋資料；LLM 負責閱讀資料並整理成自然語言回答。

Q5：什麼情況下容易產生 Hallucination？
A：當知識庫沒有相關資訊，或 Retriever 找到錯誤內容時，大型語言模型可能產生不正確或虛構的回答。

第十章 新人學習建議
建議新人依照以下順序學習：
Python 基礎。
Git 與 GitHub。
API 基本概念。
Docker 基礎。
RAG 原理。
Embedding。
Vector Database。
LangChain。
RAGFlow。
AI Agent。
透過理解上述技術，可以更容易參與企業 AI 系統的開發與維護。

"""

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50
)

chunks = splitter.split_text(text)

In [8]:
print(f"共有 {len(chunks)} 個 Chunk\n")

for i, chunk in enumerate(chunks):
    print("="*40)
    print(f"Chunk {i+1}")
    print(chunk)

共有 16 個 Chunk

Chunk 1
AI 部門新人培訓手冊（Version 1.0）
第一章 公司 AI 部門介紹
歡迎加入 AI 研發部門。
Chunk 2
第一章 公司 AI 部門介紹
歡迎加入 AI 研發部門。
本部門主要負責研究並開發企業內部 AI 應用，協助公司提升工作效率、降低重複性工作以及建立智慧化決策系統。目前部門的研究方向包括 Retrieval-Augmented Generation（RAG）、AI Agent、企業知識管理、資料分析、自動化流程以及大型語言模型（Large Language Models, LLM）。
Chunk 3
本部門希望建立一套能夠協助員工快速查詢公司知識、分析文件並回答問題的 AI 系統，因此新人需要了解 RAG 的基本概念以及相關技術。
Chunk 4
第二章 RAG 是什麼？
RAG（Retrieval-Augmented Generation）中文通常翻譯為「檢索增強生成」。
傳統的大型語言模型只能根據訓練資料回答問題，因此容易出現資訊過時或產生幻覺（Hallucination）的情況。
RAG 的核心概念是：
Chunk 5
RAG 的核心概念是：
當使用者提出問題時，系統會先到知識庫搜尋最相關的文件，再將搜尋結果提供給大型語言模型，由模型根據這些資料回答問題，而不是直接依靠模型本身的記憶。
因此，RAG 可以回答公司內部文件、產品規格、技術文件以及 SOP 等內容。
Chunk 6
第三章 RAG 的工作流程
一套完整的 RAG 系統通常包含以下步驟：
第一步，使用者將文件上傳至知識庫，例如 PDF、Word 或 PowerPoint。
第二步，系統會解析文件內容，並將文件切割成多個 Chunk。
第三步，每一個 Chunk 都會透過 Embedding Model 轉換成向量（Vector）。
第四步，向量會儲存在 Vector Database 中。
Chunk 7
第四步，向量會儲存在 Vector Database 中。
第五步，當使用者提出問題時，Retriever 會搜尋最相關的 Chunk。
最後，大型語言模型（LLM）會根據 Retriever 找到的內容產生回答。
Chunk 8
第四章 AI Agent
AI Agent 與一般聊天機器人最大的不同，在於 Agent 可以主動完

In [9]:
!pip install -q -U langchain-google-genai langchain-community faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 813.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [10]:
import langchain_google_genai
import langchain_community
import faiss

print("全部安裝成功！")

全部安裝成功！


/tmp/ipykernel_1176/2194279373.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  import langchain_community


In [11]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content=chunk,
        metadata={"chunk_id": i + 1}
    )
    for i, chunk in enumerate(chunks)
]

print(f"成功建立 {len(documents)} 個 Document")
print(documents[0])

成功建立 16 個 Document
page_content='AI 部門新人培訓手冊（Version 1.0）
第一章 公司 AI 部門介紹
歡迎加入 AI 研發部門。' metadata={'chunk_id': 1}


In [12]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=os.environ["GEMINI_API_KEY"]
)

print("Embedding 模型設定完成")

Embedding 模型設定完成


In [13]:
test_vector = embeddings.embed_query("RAG 是什麼？")

print("向量維度：", len(test_vector))
print("前 10 個數值：")
print(test_vector[:10])

向量維度： 3072
前 10 個數值：
[-0.0033329728, 0.011458252, -0.010183727, -0.058616422, -0.011289315, 0.0044718925, 0.0029071013, -0.019044416, 0.004796022, 0.0191758]


In [14]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(
    documents,
    embeddings
)

print("Vector Database 建立成功！")

Vector Database 建立成功！


In [15]:
retriever = vector_store.as_retriever()

print("Retriever 建立完成！")

Retriever 建立完成！


In [16]:
query = "RAG 是什麼？"

results = retriever.invoke(query)

print(results)

[Document(id='c366b4d3-352d-4755-ae96-8d8712517f6c', metadata={'chunk_id': 5}, page_content='RAG 的核心概念是：\n當使用者提出問題時，系統會先到知識庫搜尋最相關的文件，再將搜尋結果提供給大型語言模型，由模型根據這些資料回答問題，而不是直接依靠模型本身的記憶。\n因此，RAG 可以回答公司內部文件、產品規格、技術文件以及 SOP 等內容。'), Document(id='a8a201ba-c38b-4c5f-b000-6d94502cc3cb', metadata={'chunk_id': 4}, page_content='第二章 RAG 是什麼？\nRAG（Retrieval-Augmented Generation）中文通常翻譯為「檢索增強生成」。\n傳統的大型語言模型只能根據訓練資料回答問題，因此容易出現資訊過時或產生幻覺（Hallucination）的情況。\nRAG 的核心概念是：'), Document(id='1a26e236-35d3-4021-93bc-644ea3c88432', metadata={'chunk_id': 6}, page_content='第三章 RAG 的工作流程\n一套完整的 RAG 系統通常包含以下步驟：\n第一步，使用者將文件上傳至知識庫，例如 PDF、Word 或 PowerPoint。\n第二步，系統會解析文件內容，並將文件切割成多個 Chunk。\n第三步，每一個 Chunk 都會透過 Embedding Model 轉換成向量（Vector）。\n第四步，向量會儲存在 Vector Database 中。'), Document(id='1d5272d4-3fbb-4bf6-b811-c035a75569e3', metadata={'chunk_id': 13}, page_content='第九章 常見問題（FAQ）\nQ1：RAG 與 ChatGPT 有什麼不同？\nA：ChatGPT 主要依靠模型本身的知識回答問題；RAG 則會先搜尋知識庫，再根據搜尋結果回答，因此更適合企業內部文件查詢。\n\nQ2：Chunk 越大越好嗎？\nA：不一定。Chunk 過大可能包含過多無關資訊；Chunk 過小則可能缺乏完

In [17]:
for doc in results:
    print("=" * 50)
    print("Chunk ID:", doc.metadata["chunk_id"])
    print(doc.page_content)

Chunk ID: 5
RAG 的核心概念是：
當使用者提出問題時，系統會先到知識庫搜尋最相關的文件，再將搜尋結果提供給大型語言模型，由模型根據這些資料回答問題，而不是直接依靠模型本身的記憶。
因此，RAG 可以回答公司內部文件、產品規格、技術文件以及 SOP 等內容。
Chunk ID: 4
第二章 RAG 是什麼？
RAG（Retrieval-Augmented Generation）中文通常翻譯為「檢索增強生成」。
傳統的大型語言模型只能根據訓練資料回答問題，因此容易出現資訊過時或產生幻覺（Hallucination）的情況。
RAG 的核心概念是：
Chunk ID: 6
第三章 RAG 的工作流程
一套完整的 RAG 系統通常包含以下步驟：
第一步，使用者將文件上傳至知識庫，例如 PDF、Word 或 PowerPoint。
第二步，系統會解析文件內容，並將文件切割成多個 Chunk。
第三步，每一個 Chunk 都會透過 Embedding Model 轉換成向量（Vector）。
第四步，向量會儲存在 Vector Database 中。
Chunk ID: 13
第九章 常見問題（FAQ）
Q1：RAG 與 ChatGPT 有什麼不同？
A：ChatGPT 主要依靠模型本身的知識回答問題；RAG 則會先搜尋知識庫，再根據搜尋結果回答，因此更適合企業內部文件查詢。

Q2：Chunk 越大越好嗎？
A：不一定。Chunk 過大可能包含過多無關資訊；Chunk 過小則可能缺乏完整上下文，因此需要依照文件類型調整。


這個結果很好，代表你的 Retriever 已經正常運作，而且找出的 4 個 Chunk 都有高度相關性。

你問的是：

RAG 是什麼？

它找到：

Chunk 5：RAG 的核心概念
Chunk 4：RAG 的正式定義
Chunk 6：RAG 的工作流程
Chunk 13：RAG 與 ChatGPT 的差異

這其實是很合理的檢索結果。

為什麼順序是 5、4、6、13？

Retriever 不是照章節順序，也不是看關鍵字出現次數，而是依照向量相似度排序。

也就是：

問題「RAG 是什麼？」
→ 轉成向量
→ 和 16 個 Chunk 的向量比較
→ 找出語意最接近的前 4 個

Chunk 5 雖然不是直接寫「RAG 是什麼」，但它完整解釋了核心概念，因此在語意上可能比 Chunk 4 更接近問題。

為什麼剛好回傳 4 個？

因為：

retriever = vector_store.as_retriever()

FAISS Retriever 的預設通常是取前幾個結果，這裡實際回傳了 4 個。這個數量常稱為 Top K。

你可以明確指定：

retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)

意思是只取最相關的 3 個 Chunk。

但目前先不要改。你現在的結果已經足夠用來做完整 RAG。

In [18]:
query = "RAG 是什麼？"

results = retriever.invoke(query)

context = "\n\n".join(
    [
        f"[Chunk {doc.metadata['chunk_id']}]\n{doc.page_content}"
        for doc in results
    ]
)

prompt = f"""
你是一個企業 AI 知識助理。

請只根據下方提供的參考資料回答問題。
如果參考資料沒有足夠資訊，請回答「目前提供的資料中沒有相關資訊」。
不要使用參考資料以外的知識。

參考資料：
{context}

使用者問題：
{query}

請使用繁體中文回答，並在最後列出使用到的 Chunk 編號。
"""

print(prompt)


你是一個企業 AI 知識助理。

請只根據下方提供的參考資料回答問題。
如果參考資料沒有足夠資訊，請回答「目前提供的資料中沒有相關資訊」。
不要使用參考資料以外的知識。

參考資料：
[Chunk 5]
RAG 的核心概念是：
當使用者提出問題時，系統會先到知識庫搜尋最相關的文件，再將搜尋結果提供給大型語言模型，由模型根據這些資料回答問題，而不是直接依靠模型本身的記憶。
因此，RAG 可以回答公司內部文件、產品規格、技術文件以及 SOP 等內容。

[Chunk 4]
第二章 RAG 是什麼？
RAG（Retrieval-Augmented Generation）中文通常翻譯為「檢索增強生成」。
傳統的大型語言模型只能根據訓練資料回答問題，因此容易出現資訊過時或產生幻覺（Hallucination）的情況。
RAG 的核心概念是：

[Chunk 6]
第三章 RAG 的工作流程
一套完整的 RAG 系統通常包含以下步驟：
第一步，使用者將文件上傳至知識庫，例如 PDF、Word 或 PowerPoint。
第二步，系統會解析文件內容，並將文件切割成多個 Chunk。
第三步，每一個 Chunk 都會透過 Embedding Model 轉換成向量（Vector）。
第四步，向量會儲存在 Vector Database 中。

[Chunk 13]
第九章 常見問題（FAQ）
Q1：RAG 與 ChatGPT 有什麼不同？
A：ChatGPT 主要依靠模型本身的知識回答問題；RAG 則會先搜尋知識庫，再根據搜尋結果回答，因此更適合企業內部文件查詢。

Q2：Chunk 越大越好嗎？
A：不一定。Chunk 過大可能包含過多無關資訊；Chunk 過小則可能缺乏完整上下文，因此需要依照文件類型調整。

使用者問題：
RAG 是什麼？

請使用繁體中文回答，並在最後列出使用到的 Chunk 編號。



In [19]:
response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=prompt
)

print(response.text)

RAG（Retrieval-Augmented Generation）中文通常翻譯為「檢索增強生成」。

RAG 的核心概念是：當使用者提出問題時，系統會先到知識庫搜尋最相關的文件，再將搜尋結果提供給大型語言模型，由模型根據這些資料回答問題，而不是直接依靠模型本身的記憶。與傳統僅依賴訓練資料的大型語言模型相比，RAG 可以降低資訊過時或產生幻覺的情況，並適合用於回答公司內部文件、產品規格、技術文件以及 SOP 等內容。

使用到的 Chunk 編號：[Chunk 4]、[Chunk 5]
